# residual-skip-add — faded example 2: Choose identity vs 1x1 projection shortcut

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `residual-skip-add`. The last cell reports your progress on the `CNN: Residual skip-connection add` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Residual skip-connection add` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`residual-skip-add`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "residual-skip-add"
DD_SUBTOPIC = "CNN: Residual skip-connection add"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The shortcut is `nn.Identity()` only when shapes already match (same channels AND stride 1); otherwise it must be a 1x1 conv with the matching stride to project the input onto the residual branch's output shape.

## Faded exercise 2

The conv branch and forward are written. Complete the conditional that assigns `self.skip`: an identity when channels match and stride is 1, otherwise a 1x1 conv projection with the given stride.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch.nn as nn
Tensor = t.Tensor


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.f = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        if in_ch == out_ch and stride == 1:
            self.skip = None  # TODO: fill in this step — read the prompt cell above
        else:
            self.skip = None  # TODO: fill in this step — read the prompt cell above

    def forward(self, x: Tensor) -> Tensor:
        return self.f(x) + self.skip(x)


t.manual_seed(0)
matched = ResBlock(8, 8, stride=1)
changed = ResBlock(8, 16, stride=2)

def _test():
    matched = ResBlock(8, 8, stride=1)
    assert isinstance(matched.skip, nn.Identity), 'matched shapes -> Identity'
    changed = ResBlock(8, 16, stride=2)
    assert isinstance(changed.skip, nn.Conv2d), 'changed shapes -> Conv2d'
    assert changed.skip.kernel_size == (1, 1), 'projection must be 1x1'
    assert changed.skip.stride == (2, 2), 'projection stride must match'
    x = t.randn(2, 8, 16, 16)
    out = changed(x)
    assert tuple(out.shape) == (2, 16, 8, 8), 'projection shortcut makes the add well-defined'

try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn
Tensor = t.Tensor


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.f = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1)
        if in_ch == out_ch and stride == 1:
            self.skip = nn.Identity()
        else:
            self.skip = nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, padding=0)

    def forward(self, x: Tensor) -> Tensor:
        return self.f(x) + self.skip(x)


t.manual_seed(0)
matched = ResBlock(8, 8, stride=1)
changed = ResBlock(8, 16, stride=2)
```
</details>